In [ ]:
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import re
import os
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from tqdm.notebook import tqdm

# ==========================================
# 1. 配置 (Constants)
# ==========================================
class GameConfig:
    N_PLAYERS = 16
    EPISODE_LENGTH = 15
    ERDOS_RENYI_P = 0.3
    INITIAL_CAPITAL = 1.0
    BENEFIT_B = 0.1
    COST_C = 0.05
    PENALTY_WEIGHT_P = 1.0

class BotConfig:
    MU_THETA, SIGMA_THETA = -0.304, 2.410
    BETA_0, BETA_1, BETA_2, BETA_3 = 1.807, 0.818, 0.370, 1.521
    BETA_PRIME_0, BETA_PRIME_1 = -0.010, -0.193
    ACCEPT_PROBS = {(-1, 0): 0.774, (-1, 1): 0.085, (1, 0): 0.287, (1, 1): 0.909}

# ==========================================
# 2. 模型架构 (GraphNet)
# ==========================================
class StandardGraphNetBlock(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden_dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(nn.Linear(edge_dim + 2 * node_dim + global_dim, hidden_dim), nn.Tanh())
        self.node_mlp = nn.Sequential(nn.Linear(hidden_dim + node_dim + global_dim, hidden_dim), nn.Tanh())
        self.global_mlp = nn.Sequential(nn.Linear(hidden_dim + hidden_dim + global_dim, hidden_dim), nn.Tanh())

    def forward(self, v, e, u, mask):
        B, N, _ = v.shape
        v_s = v.unsqueeze(2).expand(-1, -1, N, -1)
        v_r = v.unsqueeze(1).expand(-1, N, -1, -1)
        u_exp = u.view(B, 1, 1, -1).expand(-1, N, N, -1)
        e_prime = self.edge_mlp(torch.cat([e, v_s, v_r, u_exp], dim=-1)) * mask.unsqueeze(-1)
        v_prime = self.node_mlp(torch.cat([e_prime.sum(dim=1), v, u.unsqueeze(1).expand(-1, N, -1)], dim=-1))
        u_prime = self.global_mlp(torch.cat([e_prime.sum(dim=(1,2)), v_prime.sum(dim=1), u], dim=-1))
        return v_prime, e_prime, u_prime

class ModifiedGraphNetBlock(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.edge_mlp = nn.Sequential(nn.Linear(input_dim + 2 * input_dim + input_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 2))
        self.node_mlp = nn.Sequential(nn.Linear(input_dim + input_dim, hidden_dim), nn.Tanh())
        self.global_mlp = nn.Sequential(nn.Linear(2 + hidden_dim + input_dim, hidden_dim), nn.Tanh(), nn.Linear(hidden_dim, 1))

    def forward(self, v, e, u, mask):
        B, N, _ = v.shape
        v_s = v.unsqueeze(2).expand(-1, -1, N, -1)
        v_r = v.unsqueeze(1).expand(-1, N, -1, -1)
        u_exp = u.view(B, 1, 1, -1).expand(-1, N, N, -1)
        e_logits = self.edge_mlp(torch.cat([e, v_s, v_r, u_exp], dim=-1)) * mask.unsqueeze(-1)
        v_prime = self.node_mlp(torch.cat([v, u.unsqueeze(1).expand(-1, N, -1)], dim=-1))
        u_value = self.global_mlp(torch.cat([e_logits.sum(dim=(1,2)), v_prime.sum(dim=1), u], dim=-1))
        return v_prime, e_logits, u_value

class SocialPlannerAgent(nn.Module):
    def __init__(self):
        super().__init__()
        h = 128
        self.block1 = StandardGraphNetBlock(2, 1, 1, h)
        self.block2 = ModifiedGraphNetBlock(h, h)

    def forward(self, capital, prev_decisions, adj, t):
        B, N = capital.shape
        v = torch.stack([capital.float(), prev_decisions.float()], dim=-1)
        e = adj.float().unsqueeze(-1)
        u = torch.full((B, 1), t / 15.0, device=capital.device)
        mask = 1.0 - torch.eye(N, device=capital.device).unsqueeze(0)
        v1, e1, u1 = self.block1(v, e, u, mask)
        _, logits, value = self.block2(v1, e1, u1, mask)
        return logits, value

# ==========================================
# 3. Bots (Rule-based & LLM)
# ==========================================
class SimulatedBots:
    def __init__(self, batch_size, device):
        self.bs, self.device, self.n = batch_size, device, GameConfig.N_PLAYERS
        self.theta = torch.normal(BotConfig.MU_THETA, BotConfig.SIGMA_THETA, (batch_size, self.n), device=device)

    def decide_cooperation(self, round_num, adj, prev_decisions, current_capital):
        x_s = adj.sum(dim=2)
        x_n = (adj * prev_decisions.unsqueeze(1)).sum(dim=2)
        x_r = torch.where(x_s > 0, x_n / x_s, torch.zeros_like(x_s))
        logits = BotConfig.BETA_PRIME_0 + BotConfig.BETA_PRIME_1 * self.theta if round_num == 0 else \
                 (BotConfig.BETA_0 + BotConfig.BETA_1 * x_s + BotConfig.BETA_2 * x_n + BotConfig.BETA_3 * x_r + self.theta)
        decisions = torch.bernoulli(torch.sigmoid(logits))
        decisions[current_capital < (GameConfig.COST_C * x_s)] = 0.0
        return decisions

    def decide_acceptance(self, recommendations, prev_decisions):
        B, N, _ = recommendations.shape
        probs = torch.zeros_like(recommendations)
        partner_prev = prev_decisions.unsqueeze(1).expand(-1, N, -1)
        for r_val in [-1, 1]:
            for p_act in [0, 1]:
                mask = (recommendations == r_val) & (partner_prev == p_act)
                probs[mask] = BotConfig.ACCEPT_PROBS[(r_val, p_act)]
        return torch.bernoulli(probs)

class LLMBots(SimulatedBots):
    def __init__(self, batch_size, device, api_key, model="gpt-4o-mini"):
        super().__init__(batch_size, device)
        self.client = OpenAI(api_key=api_key)
        self.model = model

    def _call_llm(self, prompt):
        try:
            res = self.client.chat.completions.create(model=self.model, messages=[{"role":"user","content":prompt}], temperature=0, max_tokens=2)
            return float(re.search(r'[01]', res.choices[0].message.content).group(0))
        except: return 0.0

    def decide_cooperation(self, round_num, adj, prev_decisions, current_capital):
        x_s = adj.sum(dim=2)
        x_n = (adj * prev_decisions.unsqueeze(1)).sum(dim=2)
        prompts = [f"Round {round_num+1}. Neighbors: {int(x_s[0,i])}, Cooped: {int(x_n[0,i])}. Personality: {self.theta[0,i]:.2f}. Coop(1) or Defect(0)?" for i in range(self.n)]
        with ThreadPoolExecutor(max_workers=16) as exe:
            initial = torch.tensor(list(exe.map(self._call_llm, prompts)), device=self.device).unsqueeze(0)
        initial[current_capital < (GameConfig.COST_C * x_s)] = 0.0
        return initial

# ==========================================
# 4. 环境 (Environment)
# ==========================================
class PublicGoodsGame:
    def __init__(self, batch_size, device, bots):
        self.bs, self.device, self.n, self.bots = batch_size, device, GameConfig.N_PLAYERS, bots
    
    def reset(self):
        self.current_round = 0
        rand = torch.rand(self.bs, self.n, self.n, device=self.device)
        self.adj = torch.triu((rand < GameConfig.ERDOS_RENYI_P).float(), 1)
        self.adj = self.adj + self.adj.transpose(1, 2)
        self.capital = torch.ones(self.bs, self.n, device=self.device)
        self.prev_decisions = self.bots.decide_cooperation(0, self.adj, torch.zeros_like(self.capital), self.capital)
        self._apply_payoffs(self.prev_decisions)
        return self.capital, self.prev_decisions, self.adj

    def step(self, action_logits):
        self.current_round += 1
        probs = torch.softmax(action_logits, dim=-1)[..., 1]
        change_mask = torch.bernoulli(probs) * torch.triu(torch.ones(self.n, self.n, device=self.device), 1)
        rec = torch.zeros_like(self.adj)
        rec[(self.adj == 0) & (change_mask == 1)] = 1
        rec[(self.adj == 1) & (change_mask == 1)] = -1
        accepted = self.bots.decide_acceptance(rec + rec.transpose(1,2), self.prev_decisions)
        final_change = (accepted == 1) & (change_mask == 1)
        self.adj[final_change.bool()] = 1 - self.adj[final_change.bool()]
        self.adj = torch.triu(self.adj, 1) + torch.triu(self.adj, 1).transpose(1,2)
        self.prev_decisions = self.bots.decide_cooperation(self.current_round, self.adj, self.prev_decisions, self.capital)
        self._apply_payoffs(self.prev_decisions)
        return (self.capital, self.prev_decisions, self.adj)

    def _apply_payoffs(self, coop):
        deg = self.adj.sum(dim=2)
        self.capital += (self.adj * coop.unsqueeze(1)).sum(dim=2) * GameConfig.BENEFIT_B - (GameConfig.COST_C * deg * coop)

# ==========================================
# 5. 可视化 (Visualization)
# ==========================================
def plot_game(adj, decisions, round_idx):
    G = nx.from_numpy_array(adj)
    colors = ['#4A90E2' if d == 1 else '#E74C3C' for d in decisions]
    plt.figure(figsize=(6,4))
    pos = nx.circular_layout(G)
    nx.draw(G, pos, node_color=colors, with_labels=True, node_size=500, edge_color='gray', alpha=0.7)
    plt.title(f"Round {round_idx+1}")
    plt.show()

# ==========================================
# 6. 运行测试 (Main Loop)
# ==========================================
def run_test(use_llm=False, api_key=None):
    device = torch.device("cpu")
    # 初始化 Agent (你可以尝试加载权重: agent.load_state_dict(torch.load('...')))
    agent = SocialPlannerAgent().to(device)
    agent.eval()

    # 选择 Bot
    if use_llm:
        print("Using LLM Bots...")
        bots = LLMBots(1, device, api_key)
    else:
        print("Using Rule-based Bots...")
        bots = SimulatedBots(1, device)

    env = PublicGoodsGame(1, device, bots)
    cap, dec, adj = env.reset()
    
    history_coop = [dec.mean().item()]
    plot_game(adj[0].numpy(), dec[0].numpy(), 0)

    for t in tqdm(range(GameConfig.EPISODE_LENGTH - 1)):
        logits, _ = agent(cap, dec, adj, t+1)
        cap, dec, adj = env.step(logits)
        history_coop.append(dec.mean().item())
        if (t+2) % 5 == 0 or t == GameConfig.EPISODE_LENGTH - 2:
            plot_game(adj[0].numpy(), dec[0].numpy(), t+1)
    
    plt.plot(history_coop, marker='o')
    plt.title("Cooperation Rate")
    plt.xlabel("Round")
    plt.ylabel("Rate")
    plt.grid(True)
    plt.show()



In [ ]:
run_test(use_llm=False, api_key="YOUR_OPENAI_API_KEY")